# [GENIEMUSIC] UA Incrementality Test — Result Analysis

**Ticket:** [ODSB-17054](https://mlc.atlassian.net/browse/ODSB-17054) | **Client:** Genie Music | **Analyst:** Haewon Yum  
**Test Period:** 2026-03-20 → 2026-04-08 (3 weeks)

## Objective
Measure the incremental impact of Moloco UA activity on Genie Music (지니뮤직) app installs and post-install actions using a backend ghost-bid holdout test.

## Test Design
| Field | Value |
|---|---|
| Type | Backend ghost-bid incrementality (UPLIFT) |
| App | `com.ktmusic.geniemusic` — Android, KOR |
| Campaign | UA_gpoint (`b6VKt5Rtiz6ieomx`), Goal: UA_CPA |
| Holdout | 5% ghost-bid control / 95% normal test |
| ExpLab ID | `UPLIFT-com-ktmusic-geniemusic-JsPxb` |
| Groups | Control = 16872 (50/1000 buckets) / Test = 16873 (950/1000) |
| KPI target | ≥20% incremental conversion lift |

## KPI Events (pre-test volume)
| Event | Vol/day | Priority |
|---|---|---|
| streaming_song | 714 | Primary |
| join_1 | 629 | Primary |
| gpoint_simple_success | 45 | Secondary |

## Methodology
- **Control (5%):** Ghost-bid users → organic baseline. **Test (95%):** Normal bidding → ads-influenced.
- **CPD Ratio:** Conversion-Per-Device = installs (or actions) per bucket unit.
- **Lift = CPD_test / CPD_control − 1**. Significance via 10k bootstrap resampling of 20 bins.
- ⚠️ Note: `pred_cpd_multi_metrics` only works for `GENERIC_AB_TEST` experiments. UPLIFT ghost-bid tests use bin-level bootstrap on `summary_view.experiment_summary`.

## Tables Used
- `explab-298609.summary_view.experiment_summary` — per-group daily metrics (installs, actions, spend)

In [ ]:
from google.cloud import bigquery
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
%matplotlib inline

plt.rcParams.update({'font.size': 11, 'figure.dpi': 120})
MOLOCO_BLUE   = '#1E6FFF'
MOLOCO_ORANGE = '#FF6B35'
GRAY          = '#8C8C8C'

client = bigquery.Client(project='moloco-ods')

def run_query(query, label=''):
    df = client.query(query).result().to_dataframe()
    if label:
        print(f'✅ {label}: {len(df)} rows')
    return df

print('✅ Setup complete')

In [ ]:
# ============================================================
# CASE PARAMETERS
# ============================================================
EXPERIMENT_ID   = 'UPLIFT-com-ktmusic-geniemusic-JsPxb'
EXPERIMENT_NAME = '[GENIEMUSIC] genie_KOR_UA Incrementality Test'
CLIENT_NAME     = 'Genie Music'
JIRA_TICKET     = 'ODSB-17054'

CONTROL_GROUP_ID = 16872   # 5% ghost-bid holdout (50/1000 buckets)
TEST_GROUP_ID    = 16873   # 95% normal bidding  (950/1000 buckets)

START_DATE = '2026-03-20'
END_DATE   = '2026-04-08'   # last complete day

APP_BUNDLE     = 'com.ktmusic.geniemusic'
CAMPAIGN_ID    = 'b6VKt5Rtiz6ieomx'
CAMPAIGN_NAME  = 'UA_gpoint'
CAMPAIGN_GOAL  = 'UA_CPA'
PLATFORM       = 'ANDROID'
COUNTRY        = 'KOR'

HOLDOUT_PCT         = 5.0
CONTROL_BUCKET_SIZE = 50      # out of 1000
TEST_BUCKET_SIZE    = 950
N_BINS              = 20      # number of analysis bins
KPI_LIFT_TARGET_PCT = 20.0    # client's stated target

CONTROL_SHARE = CONTROL_BUCKET_SIZE / 1000   # 0.05
TEST_SHARE    = TEST_BUCKET_SIZE    / 1000   # 0.95

print(f'📋 {CLIENT_NAME} | {EXPERIMENT_ID}')
print(f'📅 {START_DATE} → {END_DATE}')
print(f'🎯 KPI target: ≥{KPI_LIFT_TARGET_PCT:.0f}% incremental lift')

## Section 1 — Core Metrics by Group

Full metrics breakdown including bids, impressions, installs, D7 actions, and revenue for control and test groups.

In [ ]:
q_metrics = f"""
SELECT
  exp_group_id,
  CASE exp_group_id
    WHEN {CONTROL_GROUP_ID} THEN 'Control (Ghost 5%)'
    WHEN {TEST_GROUP_ID}    THEN 'Test (Normal 95%)'
  END AS group_label,
  SUM(count_bid)              AS bids,
  SUM(count_imp)              AS impressions,
  SUM(count_install)          AS installs,
  SUM(count_vt_install)       AS vt_installs,
  SUM(count_distinct_kpi_d7)  AS d7_distinct_actions,
  SUM(count_kpi_d7)           AS d7_actions,
  SUM(total_moloco_spent)     AS spend_usd,
  SUM(total_revenue_kpi_d7)   AS d7_revenue_usd
FROM `explab-298609.summary_view.experiment_summary`
WHERE utc_date BETWEEN '{START_DATE}' AND '{END_DATE}'
  AND exp_group_id IN ({CONTROL_GROUP_ID}, {TEST_GROUP_ID})
GROUP BY 1, 2
ORDER BY 1
"""
df_metrics = run_query(q_metrics, 'Core metrics by group')
display(df_metrics.T)

## Section 2 — Statistical Lift Analysis (Bootstrap CPD Ratio)

**Methodology:** Conversion-Per-Device (CPD) ratio using bin-level data (20 bins per group).  
- Normalize each bin's conversions by bucket count (control: 50/20=2.5, test: 950/20=47.5)  
- Lift = mean(CPD_test) / mean(CPD_control) − 1  
- 95% CI via bootstrap resampling (10,000 iterations)

**Why not `pred_cpd_multi_metrics`?** That function reads from `summary_v3.experiment_cumulative_summary` where `is_conducted=true`. UPLIFT ghost-bid experiments do NOT populate this table — they use a separate postback tracking pipeline.

In [ ]:
q_bins = f"""
SELECT
  exp_group_id,
  bin_number,
  SUM(count_install)         AS installs,
  SUM(count_distinct_kpi_d7) AS d7_distinct_actions,
  SUM(total_revenue_kpi_d7)  AS d7_revenue,
  SUM(count_imp)             AS impressions,
  SUM(total_moloco_spent)    AS spend_usd
FROM `explab-298609.summary_view.experiment_summary`
WHERE utc_date BETWEEN '{START_DATE}' AND '{END_DATE}'
  AND exp_group_id IN ({CONTROL_GROUP_ID}, {TEST_GROUP_ID})
GROUP BY 1, 2
ORDER BY 1, 2
"""
df_bins = run_query(q_bins, 'Bin-level data')

ctrl_bins = df_bins[df_bins.exp_group_id == CONTROL_GROUP_ID]
test_bins = df_bins[df_bins.exp_group_id == TEST_GROUP_ID]
n_bins    = len(ctrl_bins)

ctrl_bpb = CONTROL_BUCKET_SIZE / n_bins   # buckets per bin
test_bpb = TEST_BUCKET_SIZE    / n_bins

# Normalize to per-bucket CPD
ctrl_cpd     = ctrl_bins['installs'].values          / ctrl_bpb
test_cpd     = test_bins['installs'].values          / test_bpb
ctrl_cpd_act = ctrl_bins['d7_distinct_actions'].values / ctrl_bpb
test_cpd_act = test_bins['d7_distinct_actions'].values / test_bpb
ctrl_cpd_rev = ctrl_bins['d7_revenue'].values         / ctrl_bpb
test_cpd_rev = test_bins['d7_revenue'].values         / test_bpb

print(f'Bins: {n_bins} control + {n_bins} test | Control bpb={ctrl_bpb:.1f} | Test bpb={test_bpb:.1f}')
print(f'Avg CPD — Install:  ctrl={ctrl_cpd.mean():.2f}  test={test_cpd.mean():.2f}')
print(f'Avg CPD — D7 Act:   ctrl={ctrl_cpd_act.mean():.3f}  test={test_cpd_act.mean():.3f}')
print(f'Avg CPD — D7 Rev:   ctrl={ctrl_cpd_rev.mean():.2f}  test={test_cpd_rev.mean():.2f}')

In [ ]:
np.random.seed(42)

def bootstrap_ratio(ctrl_vals, test_vals, n_boot=10000, alpha=0.05):
    """Bootstrap 95% CI for CPD ratio (test/control)."""
    n = len(ctrl_vals)
    ratios = [
        np.mean(test_vals[np.random.randint(0, n, n)]) /
        max(np.mean(ctrl_vals[np.random.randint(0, n, n)]), 1e-9)
        for _ in range(n_boot)
    ]
    ratios = np.array(ratios)
    return (np.mean(ratios),
            np.percentile(ratios, alpha / 2 * 100),
            np.percentile(ratios, (1 - alpha / 2) * 100))

r_i, lb_i, ub_i = bootstrap_ratio(ctrl_cpd,     test_cpd)
r_a, lb_a, ub_a = bootstrap_ratio(ctrl_cpd_act, test_cpd_act)
r_r, lb_r, ub_r = bootstrap_ratio(ctrl_cpd_rev, test_cpd_rev)

# Structured results dict
results = {
    'CPD_INSTALL': {
        'label': 'Install Lift\n(CPD_INSTALL)',
        'ratio': r_i, 'lb': lb_i, 'ub': ub_i,
        'lift_pct': (r_i-1)*100, 'ci_lb_pct': (lb_i-1)*100, 'ci_ub_pct': (ub_i-1)*100,
        'sig': lb_i > 1.0, 'meets_target': (r_i-1)*100 >= KPI_LIFT_TARGET_PCT,
    },
    'CPD_DIST_ACT_D7': {
        'label': 'D7 Action Lift\n(CPD_DIST_ACT_D7)',
        'ratio': r_a, 'lb': lb_a, 'ub': ub_a,
        'lift_pct': (r_a-1)*100, 'ci_lb_pct': (lb_a-1)*100, 'ci_ub_pct': (ub_a-1)*100,
        'sig': lb_a > 1.0, 'meets_target': (r_a-1)*100 >= KPI_LIFT_TARGET_PCT,
    },
    'cROAS_D7': {
        'label': 'D7 Revenue Lift\n(cROAS_D7)',
        'ratio': r_r, 'lb': lb_r, 'ub': ub_r,
        'lift_pct': (r_r-1)*100, 'ci_lb_pct': (lb_r-1)*100, 'ci_ub_pct': (ub_r-1)*100,
        'sig': lb_r > 1.0, 'meets_target': (r_r-1)*100 >= KPI_LIFT_TARGET_PCT,
    },
}

print('=' * 64)
print(f'  INCREMENTALITY RESULTS — {CLIENT_NAME}  ({JIRA_TICKET})')
print(f'  {START_DATE} → {END_DATE} | 5% Ghost-Bid | {n_bins} bins | Bootstrap n=10k')
print('=' * 64)
for key, v in results.items():
    sig = '✅' if v['sig'] else '⚠️ '
    tgt = f'🎯 Meets ≥{KPI_LIFT_TARGET_PCT:.0f}%' if v['meets_target'] else f'❌ Below {KPI_LIFT_TARGET_PCT:.0f}%'
    print(f"  {key:22s}: {v['lift_pct']:+7.1f}%  95% CI [{v['ci_lb_pct']:+6.1f}%, {v['ci_ub_pct']:+6.1f}%]  {sig}  {tgt}")

## Section 3 — Interpretation & Forest Plot

In [ ]:
v_install = results['CPD_INSTALL']

if v_install['sig'] and v_install['meets_target']:
    verdict     = 'POSITIVE'
    emoji       = '✅'
    explanation = (f"Moloco demonstrates statistically significant incremental impact "
                   f"({v_install['lift_pct']:+.1f}% install lift), meeting the ≥{KPI_LIFT_TARGET_PCT:.0f}% target.")
elif v_install['sig']:
    verdict     = 'PARTIAL'
    emoji       = '⚠️'
    explanation = (f"Moloco demonstrates significant incremental impact ({v_install['lift_pct']:+.1f}% install lift), "
                   f"but below the {KPI_LIFT_TARGET_PCT:.0f}% target.")
else:
    verdict     = 'INCONCLUSIVE'
    emoji       = '❌'
    explanation = f"Lift estimate ({v_install['lift_pct']:+.1f}%) is not statistically significant."

print(f'{emoji} VERDICT: {verdict}')
print(explanation)

In [ ]:
keys   = ['CPD_INSTALL', 'CPD_DIST_ACT_D7', 'cROAS_D7']

fig, ax = plt.subplots(figsize=(11, 4))

for i, key in enumerate(keys):
    v     = results[key]
    lift  = v['lift_pct']; lb = v['ci_lb_pct']; ub = v['ci_ub_pct']
    color = MOLOCO_BLUE if v['sig'] else GRAY
    ax.barh(i, lift, height=0.45, color=color, alpha=0.85, zorder=3)
    ax.errorbar(lift, i, xerr=[[lift-lb],[ub-lift]], fmt='none',
                color='#222', capsize=5, linewidth=1.8, zorder=4)
    ax.text(max(ub,lift)+5, i, f'{lift:+.1f}%  [{lb:+.1f}%, {ub:+.1f}%]',
            va='center', fontsize=9, color='#222')

ax.axvline(0, color='#555', linewidth=0.8, alpha=0.5)
ax.axvline(KPI_LIFT_TARGET_PCT, color='red', linestyle='--', alpha=0.7,
           label=f'Target ({KPI_LIFT_TARGET_PCT:.0f}%)')
ax.set_yticks(range(len(keys)))
ax.set_yticklabels([results[k]['label'] for k in keys])
ax.set_xlabel('Incremental Lift (%, Bootstrap CPD ratio = test ÷ control)')
ax.set_title(f'Genie Music (지니뮤직) — UA Incrementality Results\n{START_DATE} to {END_DATE}  |  5% Ghost-Bid Holdout',
             fontsize=13, fontweight='bold')
ax.legend(loc='lower right'); ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.savefig('forest_plot.png', dpi=150, bbox_inches='tight')
plt.show()
print('💾 forest_plot.png')

## Section 4 — Daily Trend

In [ ]:
q_daily = f"""
SELECT utc_date, exp_group_id,
       SUM(count_imp) AS impressions, SUM(count_install) AS installs,
       SUM(total_moloco_spent) AS spend_usd
FROM `explab-298609.summary_view.experiment_summary`
WHERE utc_date BETWEEN '{START_DATE}' AND '{END_DATE}'
  AND exp_group_id IN ({CONTROL_GROUP_ID}, {TEST_GROUP_ID})
GROUP BY 1, 2 ORDER BY 1, 2
"""
df_daily = run_query(q_daily, 'Daily metrics')
df_daily['installs_norm'] = df_daily.apply(
    lambda r: r['installs'] / (CONTROL_SHARE if r['exp_group_id']==CONTROL_GROUP_ID else TEST_SHARE), axis=1
)

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 7), sharex=True)

for gid, lbl, color in [(CONTROL_GROUP_ID, 'Control (Ghost 5%)', MOLOCO_ORANGE),
                         (TEST_GROUP_ID,    'Test (Normal 95%)',  MOLOCO_BLUE)]:
    d = df_daily[df_daily.exp_group_id==gid].sort_values('utc_date')
    ax1.plot(d.utc_date, d.installs_norm, marker='o', ms=4, label=lbl, color=color, lw=1.8)

ax1.set_ylabel('Installs (normalized by bucket share)')
ax1.set_title(f'{CLIENT_NAME} — Daily Installs (Traffic-Normalized)')
ax1.legend(); ax1.grid(alpha=0.3)

td = df_daily[df_daily.exp_group_id==TEST_GROUP_ID].sort_values('utc_date')
ax2.bar(td.utc_date, td.spend_usd, color=MOLOCO_BLUE, alpha=0.7)
ax2.set_ylabel('Spend (USD)'); ax2.set_title('Test Group Daily Spend'); ax2.grid(alpha=0.3)

plt.xticks(rotation=30, ha='right'); plt.tight_layout()
plt.savefig('daily_trend.png', dpi=150, bbox_inches='tight'); plt.show()
print('💾 daily_trend.png')

## Section 5 — Result Summary Table

In [ ]:
print('=' * 64)
print(f'  FINAL VERDICT: {emoji} {verdict}')
print(f'  {explanation}')
print('=' * 64)

summary_rows = []
for key, v in results.items():
    summary_rows.append({
        'Metric':         key,
        'Lift':           f"{v['lift_pct']:+.1f}%",
        '95% CI Lower':   f"{v['ci_lb_pct']:+.1f}%",
        '95% CI Upper':   f"{v['ci_ub_pct']:+.1f}%",
        'Significant?':   '✅ Yes' if v['sig'] else '⚠️ No',
        '≥20% Target?':   '✅ Yes' if v['meets_target'] else '❌ No',
    })
display(pd.DataFrame(summary_rows))

## Section 6 — Client-Facing Slide Generation

Generates a 6-slide PPTX: Title / Test Design / Key Results / Forest Plot / Daily Trend / Recommendations

In [ ]:
%pip install python-pptx --quiet

from pptx import Presentation
from pptx.util import Inches, Pt
from pptx.dml.color import RGBColor
from pptx.enum.text import PP_ALIGN
import os
from datetime import date

C_BLUE  = RGBColor(0x1E,0x6F,0xFF); C_ORANGE= RGBColor(0xFF,0x6B,0x35)
C_DARK  = RGBColor(0x1A,0x1A,0x2E); C_WHITE = RGBColor(0xFF,0xFF,0xFF)
C_LIGHT = RGBColor(0xF5,0xF7,0xFF); C_GRAY  = RGBColor(0x8C,0x8C,0x8C)
C_GREEN = RGBColor(0x00,0xA8,0x5A); C_RED   = RGBColor(0xE5,0x3E,0x3E)
W,H = Inches(13.33),Inches(7.5)

prs = Presentation(); prs.slide_width,prs.slide_height = W,H

def add_rect(sl, l,t,w,h, fill=None, lc=None, lw=Pt(0)):
    sh = sl.shapes.add_shape(1,Inches(l),Inches(t),Inches(w),Inches(h))
    if fill: sh.fill.solid(); sh.fill.fore_color.rgb=fill
    else: sh.fill.background()
    sh.line.width=lw
    if lc: sh.line.color.rgb=lc
    else: sh.line.fill.background()
    return sh

def add_txt(sl, text, l,t,w,h, sz=16, bold=False, color=C_DARK, align=PP_ALIGN.LEFT):
    tb = sl.shapes.add_textbox(Inches(l),Inches(t),Inches(w),Inches(h))
    tb.text_frame.word_wrap=True; p=tb.text_frame.paragraphs[0]; p.alignment=align
    r=p.add_run(); r.text=text; r.font.size=Pt(sz); r.font.bold=bold; r.font.color.rgb=color
    return tb

def add_img(sl, path, l,t,w,h):
    if os.path.exists(path): sl.shapes.add_picture(path,Inches(l),Inches(t),Inches(w),Inches(h))

def blank(prs): return prs.slides.add_slide(prs.slide_layouts[6])

def footer(sl):
    add_rect(sl,0,7.2,13.33,0.3,fill=C_BLUE)
    add_txt(sl,'Moloco | Confidential',0.3,7.22,12,0.25,9,color=C_WHITE)

# S1: Title
s1=blank(prs)
add_rect(s1,0,0,13.33,7.5,fill=C_DARK); add_rect(s1,0,0,0.35,7.5,fill=C_BLUE)
add_rect(s1,0.35,5.5,13,0.06,fill=C_BLUE)
add_txt(s1,'MOLOCO',0.7,0.5,12,0.6,11,bold=True,color=C_BLUE)
add_txt(s1,'Genie Music (지니뮤직)',0.7,1.2,12,1.0,30,bold=True,color=C_WHITE)
add_txt(s1,'UA Incrementality Test — Results',0.7,2.2,12,0.7,22,color=C_WHITE)
add_txt(s1,f'Test period: {START_DATE} → {END_DATE}  |  5% Ghost-Bid Holdout  |  Android KOR',0.7,3.0,12,0.5,13,color=C_GRAY)
add_txt(s1,f'Prepared by Moloco KOR GDS  |  {date.today().strftime("%B %d, %Y")}',0.7,5.8,12,0.5,11,color=C_GRAY)

# S2: Test Design
s2=blank(prs); add_rect(s2,0,0,13.33,0.8,fill=C_DARK)
add_txt(s2,'Test Design',0.4,0.12,12,0.6,22,bold=True,color=C_WHITE)
for i,(k,v) in enumerate([('Test Type','Backend Ghost-Bid Incrementality (UPLIFT)'),
                            ('App','com.ktmusic.geniemusic  |  Android, KOR'),
                            ('Campaign',f'UA_gpoint ({CAMPAIGN_ID}) — Goal: UA_CPA'),
                            ('Holdout','5% ghost-bid control  |  95% normal test'),
                            ('Duration',f'{START_DATE} → {END_DATE}  (3 weeks)'),
                            ('KPI Target','≥20% incremental conversion lift'),
                            ('ExpLab ID',EXPERIMENT_ID)]):
    y=1.0+i*0.79
    add_rect(s2,0.4,y,2.8,0.58,fill=C_BLUE); add_txt(s2,k,0.5,y+0.1,2.6,0.45,11,bold=True,color=C_WHITE)
    add_txt(s2,v,3.4,y+0.1,9.5,0.45,11,color=C_DARK)
footer(s2)

# S3: Key Results
s3=blank(prs); add_rect(s3,0,0,13.33,0.8,fill=C_DARK)
add_txt(s3,'Key Results',0.4,0.12,12,0.6,22,bold=True,color=C_WHITE)

def kpi_box(sl, x, y, w, h, key):
    v=results[key]; lift=v['lift_pct']; lb=v['ci_lb_pct']; ub=v['ci_ub_pct']
    bc=C_BLUE if v['sig'] else RGBColor(0xBB,0xBB,0xBB)
    add_rect(sl,x,y,w,h,fill=C_LIGHT,lc=bc,lw=Pt(2))
    add_txt(sl,v['label'],x+0.1,y+0.1,w-0.2,0.5,10,bold=True,color=C_GRAY,align=PP_ALIGN.CENTER)
    add_txt(sl,f'{lift:+.1f}%',x+0.1,y+0.65,w-0.2,0.85,28,bold=True,color=C_DARK,align=PP_ALIGN.CENTER)
    add_txt(sl,f'95% CI: [{lb:+.1f}%, {ub:+.1f}%]',x+0.1,y+1.55,w-0.2,0.35,9,color=C_GRAY,align=PP_ALIGN.CENTER)
    badge='✅ Meets ≥20% target' if v['meets_target'] else '❌ Below 20% target'
    add_txt(sl,badge,x+0.1,y+1.92,w-0.2,0.35,9,bold=True,color=(C_GREEN if v['meets_target'] else C_RED),align=PP_ALIGN.CENTER)
    if not v['sig']:
        add_txt(sl,'⚠️ Not statistically significant',x+0.1,y+2.27,w-0.2,0.3,8,color=C_ORANGE,align=PP_ALIGN.CENTER)

kpi_box(s3,0.4,1.0,3.9,2.75,'CPD_INSTALL')
kpi_box(s3,4.7,1.0,3.9,2.75,'CPD_DIST_ACT_D7')
kpi_box(s3,9.0,1.0,3.9,2.75,'cROAS_D7')
banner_c=C_GREEN if verdict=='POSITIVE' else (C_ORANGE if verdict=='PARTIAL' else C_RED)
add_rect(s3,0.4,4.1,12.5,0.8,fill=banner_c)
add_txt(s3,f'{emoji}  {explanation}',0.6,4.2,12.2,0.6,12,bold=True,color=C_WHITE)
footer(s3)

# S4: Forest Plot
s4=blank(prs); add_rect(s4,0,0,13.33,0.8,fill=C_DARK)
add_txt(s4,'Statistical Confidence — Lift Estimates with 95% CI',0.4,0.12,12.5,0.6,22,bold=True,color=C_WHITE)
add_img(s4,'forest_plot.png',0.5,0.9,12.3,5.55)
add_txt(s4,'Method: Bootstrap CPD ratio (20 bins, 10k iterations). CI does not cross 0% → sig. at 95%.',0.5,6.6,12,0.4,9,color=C_GRAY)
footer(s4)

# S5: Daily Trend
s5=blank(prs); add_rect(s5,0,0,13.33,0.8,fill=C_DARK)
add_txt(s5,'Daily Performance During Test Period',0.4,0.12,12.5,0.6,22,bold=True,color=C_WHITE)
add_img(s5,'daily_trend.png',0.5,0.9,12.3,5.9); footer(s5)

# S6: Recommendations
s6=blank(prs); add_rect(s6,0,0,13.33,0.8,fill=C_DARK)
add_txt(s6,'Conclusion & Recommendations',0.4,0.12,12.5,0.6,22,bold=True,color=C_WHITE)
v_i = results['CPD_INSTALL']; v_a = results['CPD_DIST_ACT_D7']; v_r = results['cROAS_D7']
recs = [
    f'✅ Moloco UA drives +{v_i["lift_pct"]:.0f}% incremental installs (95% CI: [{v_i["ci_lb_pct"]:+.0f}%, {v_i["ci_ub_pct"]:+.0f}%]) — significantly exceeds the ≥20% target.',
    f'📈 D7 post-install actions up +{v_a["lift_pct"]:.0f}% & D7 Revenue up +{v_r["lift_pct"]:.0f}% — incremental users are high quality.',
    '💡 Recommendation: Scale UA budget. Incremental ROI is proven. Prioritize install → join_1 funnel.',
    '📅 Next: Share results with Genie Music stakeholders. Consider ROAS experiment after scaling.',
] if verdict == 'POSITIVE' else [
    f'{emoji} {explanation}',
    '🔧 Review campaign targeting, creative mix, and bid strategy.',
    '📅 Consider extending the test for more statistical power.',
    '💡 Check VT attribution configuration and KPI event setup.',
]
for i,rec in enumerate(recs):
    y=1.0+i*1.35
    add_rect(s6,0.5,y,12.3,1.1,fill=C_LIGHT,lc=C_BLUE,lw=Pt(1.5))
    add_txt(s6,rec,0.7,y+0.2,11.8,0.75,12,color=C_DARK)
footer(s6)

pptx_file = 'ODSB-17054_Genie_Music_Incrementality_Results.pptx'
prs.save(pptx_file)
print(f'💾 {pptx_file}')
print('   6 slides: Title / Test Design / Key Results / Forest Plot / Daily Trend / Recommendations')